# 🎙️ Meeting Transcription — WhisperX + pyannote Speaker Diarization

This notebook transcribes a **single mixed meeting audio file** (e.g. 3 speakers
recorded together) and produces:

| Output file | Description |
|---|---|
| `outputs/whisperx_result.json` | Full WhisperX segments + word timestamps |
| `outputs/diarization.rttm` | pyannote diarization in RTTM format |
| `outputs/speaker_transcript.txt` | Human-readable speaker-labelled transcript |
| `outputs/speaker_transcript.json` | Machine-readable list of speaker turns |

**Runtime:** GPU is strongly recommended for speed.  
Use *Runtime → Change runtime type → T4 GPU* in Colab.

**Prerequisites**
1. A Hugging Face account with an access token ([create one here](https://huggingface.co/settings/tokens)).
2. Accept the **pyannote model licences** (one-time, free):
   - https://huggingface.co/pyannote/speaker-diarization-3.1
   - https://huggingface.co/pyannote/segmentation-3.0


## Step 1 — Install Dependencies

In [ ]:
#@title 1a. System deps (ffmpeg)
%%capture
!apt-get update -qq
!apt-get install -y -qq ffmpeg
print("✅ ffmpeg installed")


In [ ]:
#@title 1b. Python deps
%%capture
# WhisperX (not on PyPI — install from GitHub)
!pip install git+https://github.com/m-bain/whisperx.git -q

# pyannote speaker diarization
!pip install pyannote.audio -q

# huggingface hub (token handling)
!pip install huggingface-hub -q

print("✅ Python dependencies installed")
print("⚠️  If this is your first install, please RESTART the Colab runtime")
print("   (Runtime → Restart session), then re-run from Step 2 onwards.")


## Step 2 — Configuration

Set your **Hugging Face token** and audio file path here.

> **Tip – Colab Secrets (recommended):** Instead of pasting your token in plain
> text, click the 🔑 icon in the left sidebar, add a secret named `HF_TOKEN`,
> and enable notebook access.  The cell below reads it automatically.


In [ ]:
#@title 2a. Hugging Face token
import os

# --- Option A: Colab Secrets (recommended) ---
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    if HF_TOKEN:
        print("✅ HF_TOKEN loaded from Colab Secrets.")
    else:
        raise KeyError("HF_TOKEN not found in secrets")
except Exception:
    # --- Option B: environment variable ---
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

# --- Option C: manual fallback (only if neither of the above worked) ---
if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass(
        "Enter your Hugging Face token (hf_…): "
    )

if not HF_TOKEN:
    raise RuntimeError(
        "No Hugging Face token found.\n"
        "Please add HF_TOKEN to Colab Secrets or set the env var."
    )

os.environ["HF_TOKEN"] = HF_TOKEN
print("HF_TOKEN is set. ✅")


In [ ]:
#@title 2b. Audio file & speaker count
import os

# ── Option A: upload a file from your computer ───────────────────────────────
# from google.colab import files
# uploaded = files.upload()
# AUDIO_PATH = list(uploaded.keys())[0]

# ── Option B: mount Google Drive and point to a file ─────────────────────────
# from google.drive import drive
# drive.mount('/content/drive')
# AUDIO_PATH = "/content/drive/MyDrive/meeting.mp3"

# ── Option C: download a sample file for testing ─────────────────────────────
AUDIO_PATH = "/content/meeting_audio.wav"
if not os.path.exists(AUDIO_PATH):
    # Download a short public-domain sample (replace with your own file)
    !wget -q -O "{AUDIO_PATH}" \
        "https://www2.cs.uic.edu/~i101/SoundFiles/gettysburg10.wav"
    print(f"Sample audio downloaded → {AUDIO_PATH}")

# ── Speaker count hint ────────────────────────────────────────────────────────
MIN_SPEAKERS = 3  #@param {type:"integer"}
MAX_SPEAKERS = 3  #@param {type:"integer"}

# ── WhisperX model size ───────────────────────────────────────────────────────
# tiny / base / small / large-v3-turbo / large-v2 / large-v3-turbo
# Use "large-v2" for best accuracy on GPU; "base" is fastest on CPU.
MODEL_SIZE = "base"  #@param ["tiny", "base", "small", "large-v3-turbo", "large-v2", "large-v3-turbo"]

print(f"Audio:       {AUDIO_PATH}")
print(f"Speakers:    {MIN_SPEAKERS}–{MAX_SPEAKERS}")
print(f"Model:       {MODEL_SIZE}")


## Step 3 — Load the `transcriptor_bot` Module

In [ ]:
#@title 3. Clone repo and set Python path
import subprocess, sys, os

REPO_URL  = "https://github.com/felipebocajr/Transcriptor-bot.git"
REPO_DIR  = "/content/Transcriptor-bot"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    print(f"✅ Repo cloned → {REPO_DIR}")
else:
    print(f"✅ Repo already present at {REPO_DIR}")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from transcriptor_bot.transcribe import run_whisperx_transcribe
from transcriptor_bot.diarize   import run_pyannote_diarization
from transcriptor_bot.assign    import (
    assign_words_to_speakers,
    build_speaker_turns,
    render_transcript,
    turns_to_json,
)
print("✅ transcriptor_bot modules imported successfully.")


## Step 4 — WhisperX Transcription + Word Alignment

This step:
1. Loads the Whisper model of the chosen size.
2. Transcribes the audio into segments with word-level timestamps.
3. Runs WhisperX alignment to get precise per-word start/end times.


In [ ]:
#@title 4. Run WhisperX
import json, os

OUT_DIR = "/content/outputs"
os.makedirs(OUT_DIR, exist_ok=True)

import torch
device       = "cuda" if torch.cuda.is_available() else "cpu"
compute_type = "float16" if device == "cuda" else "auto"
print(f"Running on: {device}  |  compute_type: {compute_type}")

whisperx_result = run_whisperx_transcribe(
    audio_path   = AUDIO_PATH,
    model_size   = MODEL_SIZE,
    device       = device,
    compute_type = compute_type,
    language     = None,   # auto-detect; set to "en" to force English
    batch_size   = 16,
)

wx_path = os.path.join(OUT_DIR, "whisperx_result.json")
with open(wx_path, "w", encoding="utf-8") as f:
    json.dump(whisperx_result, f, indent=2, ensure_ascii=False)

print(f"\n✅ WhisperX done. Detected language: {whisperx_result['language']}")
print(f"   Segments:      {len(whisperx_result['segments'])}")
print(f"   Word segments: {len(whisperx_result['word_segments'])}")
print(f"   Saved → {wx_path}")


## Step 5 — Speaker Diarization (pyannote)

pyannote segments the audio into speaker turns.  
Make sure you have accepted both model licences on the Hub (see Step 0).


In [ ]:
#@title 5. Run pyannote diarization
import os

diarization, rttm_lines = run_pyannote_diarization(
    audio_path   = AUDIO_PATH,
    hf_token     = HF_TOKEN,
    min_speakers = MIN_SPEAKERS,
    max_speakers = MAX_SPEAKERS,
)

rttm_path = os.path.join(OUT_DIR, "diarization.rttm")
with open(rttm_path, "w", encoding="utf-8") as f:
    f.write("\n".join(rttm_lines) + "\n")

print(f"✅ Diarization done.  {len(rttm_lines)} turns detected.")
print(f"   Saved → {rttm_path}")
print()
print("First 10 RTTM lines:")
for line in rttm_lines[:10]:
    print(" ", line)


## Step 6 — Assign Words to Speakers & Render Transcript

Each WhisperX word is matched to the diarization speaker with the greatest
time-overlap. Consecutive same-speaker words are then merged into turns.


In [ ]:
#@title 6. Assign & render
import json, os

assigned_words   = assign_words_to_speakers(
    word_segments = whisperx_result["word_segments"],
    diarization   = diarization,
)
turns            = build_speaker_turns(assigned_words)
transcript_text  = render_transcript(turns)

# Save plain-text transcript
txt_path = os.path.join(OUT_DIR, "speaker_transcript.txt")
with open(txt_path, "w", encoding="utf-8") as f:
    f.write(transcript_text)

# Save JSON transcript
json_path = os.path.join(OUT_DIR, "speaker_transcript.json")
with open(json_path, "w", encoding="utf-8") as f:
    f.write(turns_to_json(turns))

print(f"✅ Speaker transcript saved → {txt_path}")
print(f"✅ JSON transcript saved    → {json_path}")
print()
print("── Preview (first 30 lines) ──────────────────────────────────────────")
preview_lines = transcript_text.splitlines()[:30]
print("\n".join(preview_lines))


## Step 7 — Download Outputs

In [ ]:
#@title 7a. Zip all outputs
import shutil, os
from datetime import datetime

zip_name = f"/content/{datetime.now().strftime('%Y%m%d_%H%M%S')}_meeting_transcript"
shutil.make_archive(zip_name, "zip", OUT_DIR)
print(f"✅ Outputs zipped → {zip_name}.zip")


In [ ]:
#@title 7b. Download zip
from google.colab import files
import glob

zips = sorted(glob.glob("/content/*_meeting_transcript.zip"))
if zips:
    files.download(zips[-1])
    print(f"Downloading {zips[-1]} …")
else:
    print("No zip found — run Step 7a first.")
